
# Group 2 – Problem Analysis Workshop B

## Police-Reported Cybercrime in Canada
### K-Means Clustering Analysis

**Course:** PROG8431 – Data Analysis, Mathematics, Modeling and Algorithms

**Workshop:** Problem Analysis Workshop B

### Team Members

| Student Name | Student ID |
|---|---|
| Senay Teweldebrhan | 9120588 |
| Juan Camilo Chirivi | 9115141 |
| Zeynep Ozdemir | 9045142 |

### Project Objective

The objective of this workshop is to apply K-Means clustering to police-reported cybercrime data in Canada.

We aim to identify groups of Canadian metropolitan areas with similar cybercrime rates, explore patterns in the data, and interpret the characteristics of each cluster.


## 1. Dataset Overview
We use Statistics Canada police-reported cybercrime data.
The clustering unit is one full Census Metropolitan Area (CMA).
The values are cybercrime incidents per 100,000 population.
 
**Research question:** Which Canadian CMAs show similar reported
cybercrime rate levels in 2024 and 2025?


## 2. Setup and Configuration

We import the necessary Python libraries for data cleaning, numerical analysis, visualization, standardization, K-Means clustering, and cluster evaluation.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score



## 3. Load and Inspect the Dataset

We load the Statistics Canada cybercrime dataset and inspect its size, columns, available years, statistics, and first five rows to understand its structure.

In [3]:
file_path = Path("data/35100002.csv")
raw = pd.read_csv(file_path, low_memory=False)
print("Dataset shape:", raw.shape)
print("Columns:", list(raw.columns))
print("Years:", sorted(raw["REF_DATE"].unique()))
print("Statistics:", raw["Statistics"].unique())
display(raw.head())


Dataset shape: (1356, 15)
Columns: ['REF_DATE', 'GEO', 'DGUID', 'Statistics', 'UOM', 'UOM_ID', 'SCALAR_FACTOR', 'SCALAR_ID', 'VECTOR', 'COORDINATE', 'VALUE', 'STATUS', 'SYMBOL', 'TERMINATED', 'DECIMALS']
Years: [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Statistics: <StringArray>
['Number of cybercrime incidents', 'Rate per 100,000 population']
Length: 2, dtype: str


,REF_DATE,GEO,DGUID,Statistics,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,2014,Total police-reported cybercrime,NaN,Number of cybercrime incidents,Number,223,units,0,v116321076,1.1,15184.0,NaN,NaN,NaN,0
1,2014,Total police-reported cybercrime,NaN,"Rate per 100,000 population","Rate per 100,000 population",265,units,0,v116321077,1.2,47.7,NaN,NaN,NaN,1
2,2014,Total Non-Census metropolitan area,NaN,Number of cybercrime incidents,Number,223,units,0,v116321078,2.1,5075.0,NaN,NaN,NaN,0
3,2014,Total Non-Census metropolitan area,NaN,"Rate per 100,000 population","Rate per 100,000 population",265,units,0,v116321079,2.2,58.1,NaN,NaN,NaN,1
4,2014,Total Census metropolitan area,NaN,Number of cybercrime incidents,Number,223,units,0,v116321080,3.1,10109.0,NaN,NaN,NaN,0



## 4. Data Cleaning and Preparation

We clean the dataset by selecting cybercrime rates for 2024 and 2025, filtering Canadian Census Metropolitan Areas (CMAs), removing incomplete records, and organizing the data into two columns for K-Means clustering.

In [4]:
data = raw.copy()
data["REF_DATE"] = pd.to_numeric(data["REF_DATE"], errors="coerce")
data["VALUE"] = pd.to_numeric(data["VALUE"], errors="coerce")
 
rates = data.loc[
    data["REF_DATE"].isin([2024, 2025]) &
    data["Statistics"].eq("Rate per 100,000 population")
].copy()
 
# Five-digit geographic codes identify the CMA rows in this file.
is_cma = rates["GEO"].str.contains(r"\[\d{5}\]", regex=True, na=False)
is_split_part = rates["GEO"].str.contains(" part ", case=False, na=False)
rates = rates.loc[is_cma & ~is_split_part].copy()
 
wide = rates.pivot_table(index="GEO", columns="REF_DATE",
                         values="VALUE", aggfunc="first")
wide = wide.rename(columns={2024: "Rate_2024", 2025: "Rate_2025"})
wide = wide.dropna(subset=["Rate_2024", "Rate_2025"]).copy()
print("CMAs ready for clustering:", len(wide))
print("Missing values:", wide.isna().sum().to_dict())
display(wide.head())


CMAs ready for clustering: 39
Missing values: {'Rate_2024': 0, 'Rate_2025': 0}


REF_DATE,Rate_2024,Rate_2025
GEO,,
"Abbotsford-Mission, British Columbia [59932]",940.1,893.6
"Barrie, Ontario [35568]",158.0,114.8
"Belleville-Quinte West, Ontario [35522]",113.8,113.3
"Brantford, Ontario [35543]",301.9,430.6
"Calgary, Alberta [48825]",234.7,203.0


## 5. Why K-Means Clustering?
Workshop A used statistical comparisons. Workshop B uses unsupervised
learning to discover groups of CMAs that have similar reported cybercrime
rates without assigning groups in advance. This helps us compare rate
levels across locations and detect unusual metropolitan areas.

## 6. Two Features Used to Define the Clusters
- **Rate_2024:** Police-reported cybercrime rate per 100,000 in 2024.
- **Rate_2025:** Police-reported cybercrime rate per 100,000 in 2025.
 
Both are continuous numeric variables using the same measurement unit.
Together they reveal similarities across two consecutive years. We avoid
clustering on total incident counts because population sizes vary by CMA.
